# 01 — Simple RAG (Retrieval-Augmented Generation)

## Overview
The simplest and most foundational RAG pattern:
1. **Index** — Split documents into chunks, embed each chunk, store in a vector DB
2. **Retrieve** — Embed the user query, find the top-K most similar chunks
3. **Generate** — Feed retrieved chunks as context to Claude, get a grounded answer

## Architecture
```
User Query
    │
    ▼
Embed Query  (Amazon Bedrock Titan Embeddings V2)
    │
    ▼
Vector Search  (Qdrant Cloud  ←→  OpenSearch fallback)
    │
    ▼
Top-K Chunks
    │
    ▼
Generate Answer  (Amazon Bedrock Claude Sonnet)
    │
    ▼
Answer + Sources
```

## Vector DB Strategy
| Priority | Backend | Trigger |
|----------|---------|--------|
| 1st | **Qdrant Cloud** | `QDRANT_URL` env var is set |
| 2nd | **OpenSearch Serverless** | `OPENSEARCH_ENDPOINT` env var is set |
| 3rd | **Qdrant In-Memory** | Neither is set (dev/test fallback) |

> Set `QDRANT_URL` and `QDRANT_API_KEY` before running.

## Step 1 — Install & Import Dependencies

In [8]:
# Install required packages (safe to re-run)
import subprocess, sys
packages = [
    "boto3",
    "qdrant-client",
    "opensearch-py",
    "requests-aws4auth",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "pypdf",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


In [2]:
import os, sys, json, time, uuid
from typing import List, Dict, Optional, Tuple

import boto3
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, ScoredPoint
)

print("Imports OK")

Imports OK


## Step 2 — Configuration
Edit the values below or set them as environment variables.

In [3]:
# ── AWS / Bedrock ──────────────────────────────────────────────────────────────
AWS_REGION        = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL   = "amazon.titan-embed-text-v2:0"          # 1024-dim
LLM_MODEL         = "us.anthropic.claude-sonnet-4-6"        # Claude Sonnet 4.6

# ── Vector DB ─────────────────────────────────────────────────────────────────
QDRANT_URL        = os.getenv("QDRANT_URL",     "")          # e.g. https://xxx.qdrant.io
QDRANT_API_KEY    = os.getenv("QDRANT_API_KEY", "")          # Qdrant Cloud API key
OPENSEARCH_URL    = os.getenv("OPENSEARCH_ENDPOINT", "")     # OpenSearch endpoint

# ── RAG Parameters ────────────────────────────────────────────────────────────
COLLECTION_NAME   = "simple_rag_01"   # Qdrant collection / OpenSearch index name
EMBEDDING_DIM     = 1024              # Titan V2 output dimension
CHUNK_SIZE        = 1000              # Characters per chunk
CHUNK_OVERLAP     = 200              # Overlap between consecutive chunks
TOP_K             = 5                # Number of chunks to retrieve per query

print(f"Region         : {AWS_REGION}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model      : {LLM_MODEL}")
print(f"Qdrant URL     : {QDRANT_URL or '(not set — will use in-memory)'}")
print(f"OpenSearch URL : {OPENSEARCH_URL or '(not set)'}")
print(f"Collection     : {COLLECTION_NAME}")

Region         : us-east-1
Embedding model: amazon.titan-embed-text-v2:0
LLM model      : us.anthropic.claude-sonnet-4-6
Qdrant URL     : https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
OpenSearch URL : (not set)
Collection     : simple_rag_01


## Step 3 — Vector DB Client (Qdrant → OpenSearch fallback)

We define a thin `VectorStore` wrapper that talks Qdrant first and
falls back to OpenSearch Serverless if `QDRANT_URL` is not set.

In [4]:
class VectorStore:
    """
    Unified vector store.
    Priority: Qdrant Cloud → OpenSearch Serverless → Qdrant In-Memory
    
    Public API (same regardless of backend):
      create_collection(dim)           → bool
      upsert(docs)                     → int   docs = [{text, embedding, metadata}]
      search(query_vector, top_k)      → [{text, metadata, score, id}]
      count()                          → int
      delete_collection()              → None
    """

    def __init__(self, collection_name: str, qdrant_url: str = "",
                 qdrant_api_key: str = "", opensearch_url: str = "",
                 region: str = "us-east-1"):

        self.name   = collection_name
        self.region = region
        self._backend = None

        # ── Try Qdrant Cloud ──────────────────────────────────────────────────
        if qdrant_url:
            try:
                kwargs = {"url": qdrant_url}
                if qdrant_api_key:
                    kwargs["api_key"] = qdrant_api_key
                self._qdrant = QdrantClient(**kwargs)
                self._qdrant.get_collections()          # connectivity check
                self._backend = "qdrant_cloud"
                print(f"Connected to Qdrant Cloud: {qdrant_url}")
                return
            except Exception as e:
                print(f"Qdrant Cloud unavailable ({e}), trying next...")

        # ── Try OpenSearch Serverless ─────────────────────────────────────────
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, "aoss")
                host  = opensearch_url.replace("https://", "").replace("http://", "")
                self._os = OpenSearch(
                    hosts=[{"host": host, "port": 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30
                )
                self._os.info()                         # connectivity check
                self._backend = "opensearch"
                print(f"Connected to OpenSearch: {opensearch_url}")
                return
            except Exception as e:
                print(f"OpenSearch unavailable ({e}), falling back to in-memory Qdrant...")

        # ── Fallback: Qdrant In-Memory ────────────────────────────────────────
        self._qdrant  = QdrantClient(":memory:")
        self._backend = "qdrant_memory"
        print("Using Qdrant in-memory (data lost on kernel restart)")

    # ── Create / delete collection ────────────────────────────────────────────

    def create_collection(self, dim: int = 1024, force_recreate: bool = False) -> bool:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name)
                exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name,
                    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
                )
                print(f"Created collection '{self.name}' (dim={dim})")
            else:
                print(f"Collection '{self.name}' already exists")
            return True

        if self._backend == "opensearch":
            if force_recreate and self._os.indices.exists(index=self.name):
                self._os.indices.delete(index=self.name)
            if not self._os.indices.exists(index=self.name):
                body = {
                    "settings": {"index": {"knn": True}},
                    "mappings": {"properties": {
                        "text":      {"type": "text"},
                        "metadata":  {"type": "object"},
                        "embedding": {
                            "type": "knn_vector", "dimension": dim,
                            "method": {"name": "hnsw", "space_type": "cosinesimil",
                                       "engine": "faiss",
                                       "parameters": {"ef_construction": 512, "m": 16}}
                        }
                    }}
                }
                self._os.indices.create(index=self.name, body=body)
                print(f"Created OpenSearch index '{self.name}'")
            else:
                print(f"Index '{self.name}' already exists")
            return True

    # ── Upsert documents ──────────────────────────────────────────────────────

    def upsert(self, docs: List[Dict]) -> int:
        """docs: list of {text, embedding, metadata}"""
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            points = [
                PointStruct(
                    id=str(uuid.uuid4()),
                    vector=d["embedding"],
                    payload={"text": d["text"], "metadata": d.get("metadata", {})}
                ) for d in docs
            ]
            self._qdrant.upsert(collection_name=self.name, points=points)
            return len(points)

        if self._backend == "opensearch":
            count = 0
            for d in docs:
                self._os.index(index=self.name, body=d)
                count += 1
            time.sleep(1)   # let OpenSearch make docs searchable
            return count

    # ── Vector search ─────────────────────────────────────────────────────────

    def search(self, query_vector: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            resp = self._qdrant.query_points(
                collection_name=self.name,
                query=query_vector,
                limit=top_k,
                with_payload=True
            )
            return [
                {"text": p.payload.get("text", ""),
                 "metadata": p.payload.get("metadata", {}),
                 "score": p.score,
                 "id": str(p.id)}
                for p in resp.points
            ]

        if self._backend == "opensearch":
            body = {
                "size": top_k,
                "query": {"knn": {"embedding": {"vector": query_vector, "k": top_k}}},
                "_source": {"excludes": ["embedding"]}
            }
            resp = self._os.search(index=self.name, body=body)
            return [
                {"text": h["_source"].get("text", ""),
                 "metadata": h["_source"].get("metadata", {}),
                 "score": h["_score"],
                 "id": h["_id"]}
                for h in resp["hits"]["hits"]
            ]

    # ── Utilities ─────────────────────────────────────────────────────────────

    def count(self) -> int:
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            return self._qdrant.get_collection(self.name).points_count or 0
        if self._backend == "opensearch":
            return self._os.count(index=self.name).get("count", 0)

    def delete_collection(self):
        if self._backend in ("qdrant_cloud", "qdrant_memory"):
            self._qdrant.delete_collection(self.name)
        if self._backend == "opensearch":
            self._os.indices.delete(index=self.name, ignore=[404])
        print(f"Deleted '{self.name}'")

print("VectorStore class defined.")

VectorStore class defined.


## Step 4 — Bedrock Helpers (Embeddings + LLM)

In [5]:
# ── Single Bedrock runtime client shared across helpers ───────────────────────
bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    """
    Generate a 1024-dim embedding for a single string using
    Amazon Titan Embeddings V2 via Bedrock.
    """
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=body,
        contentType="application/json",
        accept="application/json"
    )
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str]) -> List[List[float]]:
    """
    Embed a list of texts.  Titan V2 processes one text at a time,
    so we loop with a small delay to respect rate limits.
    """
    embeddings = []
    for i, t in enumerate(texts):
        embeddings.append(embed_text(t))
        if (i + 1) % 10 == 0:
            print(f"  Embedded {i+1}/{len(texts)}")
        time.sleep(0.05)   # light rate-limit buffer
    return embeddings


def generate_answer(question: str, context_chunks: List[str]) -> str:
    """
    Call Claude via Bedrock with the retrieved chunks as context.
    Returns the model's answer as a plain string.
    """
    # Format retrieved chunks clearly so Claude can reason over them
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{chunk}" for i, chunk in enumerate(context_chunks)
    )
    prompt = (
        f"Use ONLY the context below to answer the question. "
        f"If the answer is not in the context, say 'Not found in context.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}]
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json"
    )
    return json.loads(resp["body"].read())["content"][0]["text"]


# Quick smoke test for embeddings
test_emb = embed_text("Hello world")
print(f"Embedding OK — dim={len(test_emb)}, sample={[round(x, 4) for x in test_emb[:3]]}")

Embedding OK — dim=1024, sample=[-0.0326, 0.0692, 0.0018]


## Step 5 — Connect to Vector Store & Create Collection

In [6]:
# Initialise the vector store — tries Qdrant Cloud first, then OpenSearch, then in-memory
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"\nActive backend: {vs._backend}")

# Create the collection (force_recreate=True wipes any old data from a previous run)
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

Connected to Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io

Active backend: qdrant_cloud
Created collection 'simple_rag_01' (dim=1024)


True

## Step 6 — Load & Chunk Documents

`RecursiveCharacterTextSplitter` respects paragraph/sentence boundaries before falling
back to character splits, so chunks tend to be semantically coherent.

In [7]:
# Sample corpus — replace with your own documents or load from files/PDFs
documents = [
    "Amazon Web Services (AWS) is a comprehensive cloud computing platform. "
    "It offers over 200 fully featured services including compute, storage, "
    "databases, analytics, machine learning, IoT, and more. AWS is used by "
    "millions of customers globally, from startups to Fortune 500 companies.",

    "AWS Bedrock provides access to high-performing foundation models from "
    "leading AI companies through a single API. Models available include "
    "Anthropic Claude, Amazon Titan, Meta Llama, and others. Bedrock is "
    "fully serverless — no infrastructure to manage.",

    "Qdrant is a high-performance, production-ready vector database. "
    "It is written in Rust for speed and memory efficiency. Qdrant supports "
    "filtering during vector search, making it ideal for RAG applications "
    "where metadata filters are important.",

    "Retrieval-Augmented Generation (RAG) combines information retrieval with "
    "large language model generation. A query is first used to retrieve the "
    "most relevant documents from a knowledge base, and those documents are "
    "then provided as context to the LLM to produce a grounded, accurate answer.",

    "Vector embeddings are dense numerical representations of text that capture "
    "semantic meaning. Two texts with similar meanings will have embeddings that "
    "are geometrically close to each other, enabling semantic (meaning-based) "
    "search rather than simple keyword matching.",

    "Amazon OpenSearch Serverless is a fully managed, auto-scaling deployment "
    "of OpenSearch. It removes the need to provision or manage clusters. "
    "It natively supports k-NN vector search using FAISS and HNSW, making "
    "it suitable as a vector store for RAG pipelines.",
]

# Chunk documents using LangChain's recursive splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]   # try large separators first
)

chunks = []
for doc_id, doc_text in enumerate(documents):
    for chunk_text in splitter.split_text(doc_text):
        chunks.append({"text": chunk_text, "doc_id": doc_id})

print(f"Documents loaded : {len(documents)}")
print(f"Chunks created   : {len(chunks)}")
print(f"Avg chunk length : {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")
print(f"\nSample chunk:\n  {chunks[0]['text'][:200]}")

Documents loaded : 6
Chunks created   : 6
Avg chunk length : 265 chars

Sample chunk:
  Amazon Web Services (AWS) is a comprehensive cloud computing platform. It offers over 200 fully featured services including compute, storage, databases, analytics, machine learning, IoT, and more. AWS


## Step 7 — Generate Embeddings and Index

Each chunk is embedded with Titan V2 (1024 dims) and stored in the vector store.

In [8]:
print(f"Generating embeddings for {len(chunks)} chunks...")
t0 = time.time()

# Embed all chunks
texts      = [c["text"] for c in chunks]
embeddings = embed_batch(texts)

# Build document records for upsert
docs_to_index = [
    {
        "text":      chunks[i]["text"],
        "embedding": embeddings[i],
        "metadata":  {"doc_id": chunks[i]["doc_id"], "chunk_index": i}
    }
    for i in range(len(chunks))
]

# Upsert into the vector store
indexed = vs.upsert(docs_to_index)
elapsed = time.time() - t0

print(f"\nIndexed  : {indexed} chunks")
print(f"Time     : {elapsed:.2f}s  ({elapsed/len(chunks):.2f}s per chunk)")
print(f"Count in store: {vs.count()}")

Generating embeddings for 6 chunks...



Indexed  : 6 chunks
Time     : 1.15s  (0.19s per chunk)
Count in store: 6


## Step 8 — Query the RAG System

For each test question:
1. Embed the question with Titan
2. Retrieve top-K chunks from the vector store
3. Pass chunks + question to Claude → get answer

In [9]:
def rag_query(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    """
    Full RAG pipeline for a single question.
    Returns: {question, answer, sources, latency_ms}
    """
    t0 = time.time()

    # Step A: embed the question
    q_embedding = embed_text(question)

    # Step B: retrieve top-K chunks
    results = vs.search(q_embedding, top_k=top_k)

    # Step C: generate answer using retrieved chunks as context
    context_texts = [r["text"] for r in results]
    answer = generate_answer(question, context_texts)

    latency_ms = (time.time() - t0) * 1000

    if verbose:
        print(f"\nQuestion : {question}")
        print(f"Latency  : {latency_ms:.0f}ms")
        print(f"\nAnswer:\n{answer}")
        print(f"\nTop sources retrieved:")
        for i, r in enumerate(results[:3], 1):
            print(f"  [{i}] score={r['score']:.4f}  {r['text'][:120]}...")
        print("-" * 70)

    return {"question": question, "answer": answer,
            "sources": results, "latency_ms": latency_ms}


# Run test questions
test_questions = [
    "What is AWS Bedrock and which models does it support?",
    "How does RAG work?",
    "Why is Qdrant written in Rust?",
    "What is the difference between semantic search and keyword search?",
]

results_log = []
for q in test_questions:
    r = rag_query(q)
    results_log.append(r)


Question : What is AWS Bedrock and which models does it support?
Latency  : 3088ms

Answer:
## AWS Bedrock

Based on the provided context, **AWS Bedrock** is a service that provides access to **high-performing foundation models** from leading AI companies through a **single API**. It is **fully serverless**, meaning there is no infrastructure to manage.

### Supported Models:
The following models are available through AWS Bedrock:
- **Anthropic Claude**
- **Amazon Titan**
- **Meta Llama**
- And **others** (additional models are mentioned but not specified in the context)

> *Source: Chunk 1*

Top sources retrieved:
  [1] score=0.8622  AWS Bedrock provides access to high-performing foundation models from leading AI companies through a single API. Models ...
  [2] score=0.3667  Amazon Web Services (AWS) is a comprehensive cloud computing platform. It offers over 200 fully featured services includ...
  [3] score=0.1952  Amazon OpenSearch Serverless is a fully managed, auto-scaling deploy


Question : How does RAG work?
Latency  : 2725ms

Answer:
Based on the context provided, RAG (Retrieval-Augmented Generation) works in the following way:

1. **Retrieval step**: A query is first used to retrieve the most relevant documents from a knowledge base.
2. **Generation step**: Those retrieved documents are then provided as context to a Large Language Model (LLM).
3. **Output**: The LLM uses that context to produce a **grounded and accurate answer**.

In essence, RAG combines **information retrieval** with **LLM-based generation** to improve the accuracy and reliability of AI-generated responses.

Top sources retrieved:
  [1] score=0.6724  Retrieval-Augmented Generation (RAG) combines information retrieval with large language model generation. A query is fir...
  [2] score=0.1723  Qdrant is a high-performance, production-ready vector database. It is written in Rust for speed and memory efficiency. Q...
  [3] score=0.1064  Amazon OpenSearch Serverless is a fully managed, auto-sc


Question : Why is Qdrant written in Rust?
Latency  : 1321ms

Answer:
Based on the context, Qdrant is written in Rust **for speed and memory efficiency**.

Top sources retrieved:
  [1] score=0.6986  Qdrant is a high-performance, production-ready vector database. It is written in Rust for speed and memory efficiency. Q...
  [2] score=0.1112  Retrieval-Augmented Generation (RAG) combines information retrieval with large language model generation. A query is fir...
  [3] score=0.0941  AWS Bedrock provides access to high-performing foundation models from leading AI companies through a single API. Models ...
----------------------------------------------------------------------



Question : What is the difference between semantic search and keyword search?
Latency  : 2622ms

Answer:
Based on the context provided, **semantic search** is meaning-based search that uses **vector embeddings** — dense numerical representations of text that capture semantic meaning — where texts with similar meanings have embeddings that are geometrically close to each other.

By contrast, **keyword search** is described simply as "simple keyword matching."

The key difference is that semantic search understands the *meaning* behind text, while keyword search relies on literal word matching.

Top sources retrieved:
  [1] score=0.2186  Vector embeddings are dense numerical representations of text that capture semantic meaning. Two texts with similar mean...
  [2] score=0.1805  Amazon OpenSearch Serverless is a fully managed, auto-scaling deployment of OpenSearch. It removes the need to provision...
  [3] score=0.0616  Amazon Web Services (AWS) is a comprehensive cloud computing platfo

## Step 9 — Evaluation & Metrics

In [10]:
# ── Lightweight evaluation: keyword coverage + latency ────────────────────────
eval_cases = [
    {"question": "What is AWS Bedrock and which models does it support?",
     "expected_keywords": ["foundation", "Claude", "Titan", "API", "serverless"]},
    {"question": "How does RAG work?",
     "expected_keywords": ["retrieval", "context", "generation", "query", "document"]},
    {"question": "Why is Qdrant written in Rust?",
     "expected_keywords": ["Rust", "speed", "memory"]},
]

print(f"{'Question':<55} {'KW Hit':>7} {'Latency':>10}")
print("-" * 75)

for case in eval_cases:
    result = rag_query(case["question"], verbose=False)
    answer_lower = result["answer"].lower()
    hits   = sum(1 for kw in case["expected_keywords"] if kw.lower() in answer_lower)
    total  = len(case["expected_keywords"])
    pct    = hits / total * 100
    lat    = result["latency_ms"]
    print(f"{case['question'][:54]:<55} {hits}/{total} ({pct:.0f}%) {lat:>8.0f}ms")

print()
avg_lat = sum(r["latency_ms"] for r in results_log) / len(results_log)
print(f"Average latency across all queries: {avg_lat:.0f}ms")

Question                                                 KW Hit    Latency
---------------------------------------------------------------------------


What is AWS Bedrock and which models does it support?   5/5 (100%)     2339ms


How does RAG work?                                      5/5 (100%)     2666ms


Why is Qdrant written in Rust?                          3/3 (100%)     1430ms

Average latency across all queries: 2439ms


## Step 10 — Summary

### What we built
| Component | Implementation |
|-----------|---------------|
| **Chunking** | `RecursiveCharacterTextSplitter` (1000 chars, 200 overlap) |
| **Embeddings** | Amazon Bedrock Titan V2 — 1024 dims |
| **Vector DB** | Qdrant Cloud (falls back to OpenSearch → in-memory) |
| **LLM** | Amazon Bedrock Claude Sonnet 4.6 |
| **Retrieval** | Cosine similarity, top-5 |

### Strengths
- Simple and fast to implement
- Good baseline for comparison with advanced patterns
- Works well for straightforward Q&A

### Limitations
- No reranking — top-K may include low-quality matches
- Fixed chunk size — may cut across important context boundaries
- Single query — no query expansion or reformulation

### Next: **02 — Graph RAG** (entities + relationships, multi-hop queries)

In [11]:
# Optional cleanup — uncomment to delete the Qdrant collection after testing
# vs.delete_collection()

print(f"Collection '{COLLECTION_NAME}' retained in {vs._backend}.")
print(f"Total vectors stored: {vs.count()}")
print("\nDone. Give the go-ahead to proceed to notebook 02.")

Collection 'simple_rag_01' retained in qdrant_cloud.
Total vectors stored: 6

Done. Give the go-ahead to proceed to notebook 02.
